In [16]:
import pandas as pd

station = pd.read_parquet("../../Data/sort_data/2024_data.parquet")

In [17]:
# 특정 정류소가 시작이거나 종료인 데이터만 반환
def filter_station_rows(df, keyword):
    mask = (
        df['시작_대여소_ID'].astype(str).str.contains(keyword, na=False) |
        df['종료_대여소_ID'].astype(str).str.contains(keyword, na=False)
    )
    
    filtered_df = df[mask].copy()
    
    return filtered_df

In [18]:
# 종료가 해당 정류소인 경우 도착시간으로 변경
import numpy as np

def update_end(df, station_id):
    mask = df["종료_대여소_ID"] == station_id

    # Timedelta -> 시간(실수)
    hours = df.loc[mask, "전체_이용_분"].dt.total_seconds() / 3600.0

    # 버림 처리
    raw = np.floor(df.loc[mask, "시간대"] - hours).astype(int)

    # 음수 시간 처리: 기준_날짜 하루(또는 여러 일) 줄이고 시간대 보정
    day_shift = np.floor_divide(raw, 24)  # 음수면 -1, -2 ...
    adj_hour = raw - day_shift * 24       # 0~23로 보정

    # 날짜 업데이트
    df.loc[mask, "기준_날짜"] = (
        pd.to_datetime(df.loc[mask, "기준_날짜"]) + pd.to_timedelta(day_shift, unit="D")
    )

    # 시간대 업데이트
    df.loc[mask, "시간대"] = adj_hour

    # 집계 기준 변경
    df.loc[mask, "집계_기준"] = "도착시간"

    # 기준_날짜 -> 시간대 정렬
    df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)
    return df


In [19]:
# 출퇴근시간, 요일, 주말 컬럼 추가
def add_time_flags(df, commute_hours=None):
    df = df.copy()

    # 출퇴근 시간대 기본값
    if commute_hours is None:
        commute_hours = [7, 8, 9, 17, 18, 19]

    # 날짜 파싱
    date = pd.to_datetime(df["기준_날짜"])

    # 요일 컬럼 (월~일)
    weekday_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
    df["요일"] = date.dt.weekday.map(weekday_map)

    # 출퇴근시간 컬럼
    df["출퇴근시간"] = df["시간대"].isin(commute_hours).astype(int)

    # 2024년 공휴일 (한국)
    holidays_2024 = pd.to_datetime([
        "2024-01-01",
        "2024-02-09", "2024-02-10", "2024-02-11", "2024-02-12",
        "2024-03-01",
        "2024-04-10",
        "2024-05-05", "2024-05-06",
        "2024-05-15",
        "2024-06-06",
        "2024-08-15",
        "2024-09-16", "2024-09-17", "2024-09-18",
        "2024-10-01",
        "2024-10-03",
        "2024-10-09",
        "2024-12-25",
    ])

    # 주말 컬럼 (토/일 또는 2024 공휴일)
    is_weekend = date.dt.weekday >= 5
    is_holiday_2024 = date.isin(holidays_2024)
    df["주말"] = (is_weekend | is_holiday_2024).astype(int)

    return df


In [20]:
st_2247 = filter_station_rows(station, "ST-2247")
st_2252 = filter_station_rows(station, "ST-2252")
st_2425 = filter_station_rows(station, "ST-2425")

In [21]:
st_2247 = update_end(st_2247, "ST-2247")
st_2252 = update_end(st_2252, "ST-2252")
st_2425 = update_end(st_2425, "ST-2425")

/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_9519/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[13 18 17 ... 12 13  6]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_9519/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[16 16 16 ... 14 20 13]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_9519/3903656530.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[15 15 15 ...  0 18 10]' has dtype incompatible with int32, please explicitly

In [22]:
st_2247 = add_time_flags(st_2247)
st_2252 = add_time_flags(st_2252)
st_2425 = add_time_flags(st_2425)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_model(
    df,
    time_col=("기준_날짜", "시간대"),
    feature_cols=("시간대","주말", "출퇴근시간", "온도", "습도", "강수량", "적설량"),
    target_col="전체_건수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
):
    # 1) 정렬용 컬럼 준비
    df_sorted = df.copy()
    if "기준_날짜" in time_col:
        df_sorted["기준_날짜"] = pd.to_datetime(df_sorted["기준_날짜"])

    # 2) 기준_날짜 + 시간대로 정렬
    df_sorted = df_sorted.sort_values(list(time_col)).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 3) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 4) 파이프라인 (정규화 + 랜덤포레스트)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1))
    ])

    # 5) 하이퍼파라미터 탐색
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 12],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", "log2"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # 6) 점수 계산
    def _eval(name, X_split, y_split):
        pred = best_model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": np.sqrt(mean_squared_error(y_split, pred))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test),
        "best_params": search.best_params_
    }

    return best_model, scores


In [26]:
train_model(st_2247)

(Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  RandomForestRegressor(max_depth=8, max_features='log2',
                                        min_samples_leaf=2, n_estimators=200,
                                        n_jobs=-1, random_state=42))]),
 {'train': {'split': 'train',
   'r2': 0.07352014785842864,
   'mae': 0.049158604105653994,
   'rmse': np.float64(0.16294380390143756)},
  'val': {'split': 'val',
   'r2': -0.0008513724128171862,
   'mae': 0.060750794571215745,
   'rmse': np.float64(0.19302236840179895)},
  'test': {'split': 'test',
   'r2': -0.0049479862293038135,
   'mae': 0.04880800055560636,
   'rmse': np.float64(0.17353092174727486)},
  'best_params': {'model__max_depth': 8,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 2,
   'model__n_estimators': 200}})